In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from gravity3d_variable_density import compute_gravity

# --- Load gravity anomaly data ---
data = pd.read_csv("gravity_anomaly_with_coords_1.csv")
x = data['X(m)'].values
y = data['Y(m)'].values
gz_obs = data['GravityAnomaly(mGal)'].values.reshape(15, 15)

# --- Grid setup ---
Lx = Ly = 5000
nx = ny = 15
nz = 20
max_depth = 150.0

x_edges = np.linspace(0, Lx, nx + 1)
y_edges = np.linspace(0, Ly, ny + 1)
z_edges = np.linspace(0, max_depth, nz + 1)

# Observation grid (flat at surface)
x_obs = np.linspace(0, Lx, nx)
y_obs = np.linspace(0, Ly, ny)
Xobs, Yobs = np.meshgrid(x_obs, y_obs, indexing='ij')
Zobs = np.zeros_like(Xobs)

# --- Depth-dependent density model parameters ---
rho_min = 1500.0  # kg/m³
rho_max = 3500.0  # kg/m³
alpha = 0.02      # 1/m

def depth_based_density(z):
    """
    Continuous density model increasing with depth.
    """
    return rho_min + (rho_max - rho_min) * (1 - np.exp(-alpha * z))

def build_density_model(depth_surface):
    """
    Generate 3D density model based on exponential depth-dependent law.
    Only voxels above basin surface are filled.
    """
    rho_model = np.zeros((nx, ny, nz), dtype=np.float32)
    for i in range(nx):
        for j in range(ny):
            zb = depth_surface[i, j]
            for k in range(nz):
                z1, z2 = z_edges[k], z_edges[k + 1]
                zc = 0.5 * (z1 + z2)
                if zc <= zb:
                    rho_model[i, j, k] = depth_based_density(zc)
                else:
                    rho_model[i, j, k] = rho_max
    return rho_model

def objective(zb_flat):
    """
    Objective function for inversion (RMSE).
    """
    zb = zb_flat.reshape((nx, ny))
    rho_model = build_density_model(zb)
    gz_model = compute_gravity(Xobs, Yobs, Zobs, x_edges, y_edges, z_edges, rho_model)
    rmse = np.sqrt(np.mean((gz_obs - gz_model) ** 2))
    return rmse

# --- Inversion bounds: depth between 0 and max_depth ---
bounds = [(0.0, max_depth)] * (nx * ny)

# --- Run inversion using Differential Evolution ---
result = differential_evolution(objective, bounds, strategy='best1bin',
                                maxiter=80, popsize=15, tol=1e-3,disp=True)

# --- Extract inverted basin surface ---
zb_inverted = result.x.reshape((nx, ny))
rho_final = build_density_model(zb_inverted)
gz_final = compute_gravity(Xobs, Yobs, Zobs, x_edges, y_edges, z_edges, rho_final)

# --- Plot Results ---
plt.figure(figsize=(16, 4))

plt.subplot(1, 3, 1)
plt.title("Observed Gravity (mGal)")
plt.contourf(Xobs, Yobs, gz_obs, cmap='viridis')
plt.colorbar(label="mGal")

plt.subplot(1, 3, 2)
plt.title("Modeled Gravity (mGal)")
plt.contourf(Xobs, Yobs, gz_final, cmap='viridis')
plt.colorbar(label="mGal")

plt.subplot(1, 3, 3)
plt.title("Inverted Basin Depth (m)")
plt.contourf(Xobs, Yobs, zb_inverted, cmap='plasma')
plt.colorbar(label="Depth (m)")

plt.tight_layout()
plt.show()


KeyboardInterrupt: 